In [3]:
import pandas as pd

In [4]:
return_average = pd.read_csv("0.003_benchmark.csv", header = 0, index_col = 0)

In [5]:
return_average

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
2019-01-30,0.060012,0.041212,-0.079102,-0.016721,0.001350,0.054127
2019-02-28,0.020955,0.012108,0.026901,0.012351,0.018079,0.019223
2019-03-28,-0.005156,-0.010123,-0.008995,-0.011004,-0.008819,-0.003245
2019-04-26,0.027753,0.005332,0.021694,0.021245,0.019006,0.027546
2019-05-24,-0.027919,-0.013556,-0.030867,-0.034561,-0.026726,-0.031184
...,...,...,...,...,...,...
2024-09-20,0.030146,0.041552,0.007695,-0.025107,0.013572,0.034196
2024-10-18,0.007070,0.013938,0.026079,0.014277,0.015341,0.001142
2024-11-15,-0.043323,-0.028632,-0.006021,0.006907,-0.017767,-0.016485
2024-12-16,0.018710,0.022765,0.065300,0.083412,0.047547,0.037941


In [6]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [7]:
from scipy.stats import t


In [19]:
for st in return_average.keys():
    print(st)
    desc = return_average[st].describe()
    std_error = desc["std"] / (desc["count"] ** 0.5)
    print(std_error)
    print("skewness:", return_average[st].skew()
          , "kurtosis:", return_average[st].kurtosis())

risk_parity
0.004880917996684978
skewness: -0.896795632907797 kurtosis: 3.1187155213685207
min_var
0.0037778319357243896
skewness: -0.40774272617118806 kurtosis: 1.1220662351530724
max_sharpe
0.0069746715805717166
skewness: -0.1854722148147326 kurtosis: 1.9162983794642612
paa
0.005726011994325484
skewness: -0.6656643337041376 kurtosis: 1.9151218310897002
equal_strategy
0.004494855108929811
skewness: -0.9358872038864138 kurtosis: 2.8102394682394545
equal_asset_weight
0.005206775338797625
skewness: -0.7526029961252102 kurtosis: 2.5317594601022755


In [14]:
return_average.describe()

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
count,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000
mean,0.006316,0.010279,0.007207,0.009076,0.008219,0.006322
std,0.042551,0.032934,0.060804,0.049918,0.039185,0.045392
min,-0.165434,-0.098912,-0.162176,-0.156020,-0.128480,-0.146382
25%,-0.015096,-0.009077,-0.020255,-0.014617,-0.009471,-0.011593
50%,0.007588,0.012823,0.006781,0.012222,0.013587,0.009150
75%,0.030126,0.030572,0.035427,0.037472,0.031178,0.033506
max,0.099199,0.085346,0.191945,0.129454,0.090588,0.126259


In [9]:
return_average.columns

Index(['risk_parity', 'min_var', 'max_sharpe', 'paa', 'equal_strategy',
       'equal_asset_weight'],
      dtype='object')

In [13]:
imp = pd.DataFrame(index = return_average.columns)

for col in return_average.columns:
    
    # 계산 실행
    mean_return, nw_se, nw_tstat = newey_west_tstat(return_average[col], maxlags=12)



    # 단측 검정 (우측): P(T > t)
    p_value = 1 - t.cdf(nw_tstat, df=len(return_average))
    print(f"Column: {col}")
    print(f"p-value = {p_value:.6f}")
    print(f"Mean Return = {mean_return:.6f}, NW SE = {nw_se:.6f}, NW t-stat = {nw_tstat:.6f}")
    imp.loc[col, "mean_return"] = mean_return
    imp.loc[col, "nw_se"] = nw_se
    imp.loc[col, "nw_tstat"] = nw_tstat
    imp.loc[col, "p_value"] = p_value


Column: risk_parity
p-value = 0.080954
Mean Return = 0.006316, NW SE = 0.004472, NW t-stat = 1.412423
Column: min_var
p-value = 0.003062
Mean Return = 0.010279, NW SE = 0.003645, NW t-stat = 2.819912
Column: max_sharpe
p-value = 0.226937
Mean Return = 0.007207, NW SE = 0.009573, NW t-stat = 0.752835
Column: paa
p-value = 0.069562
Mean Return = 0.009076, NW SE = 0.006072, NW t-stat = 1.494739
Column: equal_strategy
p-value = 0.056297
Mean Return = 0.008219, NW SE = 0.005120, NW t-stat = 1.605211
Column: equal_asset_weight
p-value = 0.085045
Mean Return = 0.006322, NW SE = 0.004564, NW t-stat = 1.385050


In [20]:
imp.T

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
mean_return,0.006079,0.008146,0.008445,0.003935,0.006651,0.006390
nw_se,0.004850,0.003728,0.007207,0.007782,0.004968,0.005113
nw_tstat,1.253409,2.185118,1.171847,0.505658,1.338731,1.249568
p_value,0.106950,0.015981,0.122460,0.307280,0.092324,0.107646


In [31]:
nw_tstat

np.float64(1.2891250406400871)

In [25]:
return_average

,date,return
0,2019-01-30,0.001730
1,2019-02-28,0.027784
2,2019-03-28,-0.006986
3,2019-04-26,0.019037
4,2019-05-24,-0.033522
...,...,...
71,2024-09-20,0.009221
72,2024-10-18,0.022761
73,2024-11-15,-0.013343
74,2024-12-16,0.106342


In [26]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.05455205575)